# 04 — LSTM Model
**EduForecast** | Sequential dropout prediction | Notebook 4 of 6

In [1]:
import copy
import random
import sys
import time
import warnings
from pathlib import Path

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np
import pandas as pd
import seaborn as sns
import torch
import torch.nn as nn
from sklearn.metrics import (
    accuracy_score, classification_report,
    f1_score, precision_score, recall_score,
    roc_auc_score, roc_curve,
)
from sklearn.preprocessing import StandardScaler
from torch.utils.data import DataLoader, TensorDataset

warnings.filterwarnings('ignore')

PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.config import PROCESSED_DIR, FIGURES_DIR, METRICS_DIR, RESULTS_DIR, RANDOM_SEED

# Reproducibility
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)
if torch.backends.mps.is_available():
    torch.mps.manual_seed(RANDOM_SEED)

sns.set_style('whitegrid')
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
METRICS_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
DPI = 150

print('Environment ready.')
print(f'PyTorch version : {torch.__version__}')
print(f'MPS available   : {torch.backends.mps.is_available()}')
print(f'CUDA available  : {torch.cuda.is_available()}')

Environment ready.
PyTorch version : 2.12.0
MPS available   : True
CUDA available  : False


## 1. Load Processed Data

In [2]:
print('Loading sequential feature files...')

train_df = pd.read_csv(PROCESSED_DIR / 'train_sequential.csv')
test_df  = pd.read_csv(PROCESSED_DIR / 'test_sequential.csv')

print(f'train_sequential : {train_df.shape}')
print(f'test_sequential  : {test_df.shape}')

REG_KEYS  = ['id_student', 'code_module', 'code_presentation']
FEAT_COLS = [
    'weekly_clicks', 'is_active', 'cumulative_clicks',
    'click_trend', 'weeks_since_active',
    'assessment_score', 'assessment_submitted',
]
TARGET   = 'is_at_risk'
N_WEEKS  = 40
N_FEAT   = len(FEAT_COLS)

print(f'\nFeature columns ({N_FEAT}): {FEAT_COLS}')
print(f'Weeks per student: {N_WEEKS}')

Loading sequential feature files...


train_sequential : (1042400, 12)
test_sequential  : (261320, 12)

Feature columns (7): ['weekly_clicks', 'is_active', 'cumulative_clicks', 'click_trend', 'weeks_since_active', 'assessment_score', 'assessment_submitted']
Weeks per student: 40


## 2. Prepare Sequential Data

In [3]:
print('Reshaping into 3-D tensors...')

# Sort so every registration occupies exactly 40 contiguous rows (weeks 0-39)
train_df = train_df.sort_values(REG_KEYS + ['week']).reset_index(drop=True)
test_df  = test_df.sort_values(REG_KEYS + ['week']).reset_index(drop=True)

# Verify: every registration has exactly 40 rows
for split, df in [('train', train_df), ('test', test_df)]:
    counts = df.groupby(REG_KEYS).size()
    assert (counts == N_WEEKS).all(), f'Not all registrations have {N_WEEKS} weeks in {split}!'
    n_reg = len(counts)
    print(f'{split}: {n_reg:,} registrations × {N_WEEKS} weeks = {n_reg*N_WEEKS:,} rows \u2713')

n_train = len(train_df) // N_WEEKS
n_test  = len(test_df)  // N_WEEKS

# Labels: is_at_risk is identical for all 40 rows of a registration.
# After the sort, every 40th row (stride=N_WEEKS) is week 0 of a new registration.
y_train = train_df.iloc[::N_WEEKS][TARGET].values.astype(np.float32)
y_test  = test_df.iloc[::N_WEEKS][TARGET].values.astype(np.float32)

# Features → 3-D arrays (registrations, weeks, features)
X_train_raw = train_df[FEAT_COLS].values.reshape(n_train, N_WEEKS, N_FEAT).astype(np.float32)
X_test_raw  = test_df[FEAT_COLS].values.reshape(n_test,  N_WEEKS, N_FEAT).astype(np.float32)

# Sanity check: week 0 should be the first week of each registration block
assert (train_df.iloc[::N_WEEKS]['week'] == 0).all(), 'Week 0 alignment mismatch!'

print(f'\nX_train shape : {X_train_raw.shape}')
print(f'X_test  shape : {X_test_raw.shape}')
print(f'y_train shape : {y_train.shape}  at-risk: {y_train.mean()*100:.1f}%')
print(f'y_test  shape : {y_test.shape}   at-risk: {y_test.mean()*100:.1f}%')

Reshaping into 3-D tensors...


train: 26,060 registrations × 40 weeks = 1,042,400 rows ✓
test: 6,533 registrations × 40 weeks = 261,320 rows ✓

X_train shape : (26060, 40, 7)
X_test  shape : (6533, 40, 7)
y_train shape : (26060,)  at-risk: 31.2%
y_test  shape : (6533,)   at-risk: 31.2%


In [4]:
# Scale features: fit on train 2-D, transform both, reshape back to 3-D
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(
    X_train_raw.reshape(-1, N_FEAT)
).reshape(n_train, N_WEEKS, N_FEAT).astype(np.float32)

X_test_sc = scaler.transform(
    X_test_raw.reshape(-1, N_FEAT)
).reshape(n_test, N_WEEKS, N_FEAT).astype(np.float32)

assert not np.isnan(X_train_sc).any(), 'NaN in scaled X_train'
assert not np.isnan(X_test_sc).any(),  'NaN in scaled X_test'
print('Scaling complete. No NaN values.')

# PyTorch tensors
X_train_t = torch.from_numpy(X_train_sc)
y_train_t = torch.from_numpy(y_train)
X_test_t  = torch.from_numpy(X_test_sc)
y_test_t  = torch.from_numpy(y_test)

# DataLoaders
BATCH_SIZE = 64
train_loader = DataLoader(
    TensorDataset(X_train_t, y_train_t),
    batch_size=BATCH_SIZE, shuffle=True,
    generator=torch.Generator().manual_seed(RANDOM_SEED),
)
test_loader = DataLoader(
    TensorDataset(X_test_t, y_test_t),
    batch_size=BATCH_SIZE, shuffle=False,
)

print(f'\nDataLoaders ready.')
print(f'  train batches : {len(train_loader)}')
print(f'  test  batches : {len(test_loader)}')
print(f'  batch size    : {BATCH_SIZE}')

Scaling complete. No NaN values.

DataLoaders ready.
  train batches : 408
  test  batches : 103
  batch size    : 64


## 3. Define LSTM Model

In [5]:
class StudentLSTM(nn.Module):
    """2-layer LSTM binary classifier for student dropout prediction.

    Outputs raw logits (no sigmoid) — use BCEWithLogitsLoss during training
    and torch.sigmoid() during inference.
    """

    def __init__(
        self,
        input_size: int = 7,
        hidden_size: int = 64,
        num_layers: int = 2,
        dropout: float = 0.2,
    ) -> None:
        super().__init__()
        self.lstm = nn.LSTM(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0.0,
        )
        self.classifier = nn.Sequential(
            nn.Linear(hidden_size, 32),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(32, 1),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """x: (batch, seq_len, input_size) -> logits: (batch,)"""
        lstm_out, _ = self.lstm(x)       # (batch, seq, hidden)
        last = lstm_out[:, -1, :]        # last timestep: (batch, hidden)
        return self.classifier(last).squeeze(-1)  # (batch,)


model_probe = StudentLSTM()
total_params     = sum(p.numel() for p in model_probe.parameters())
trainable_params = sum(p.numel() for p in model_probe.parameters() if p.requires_grad)

print('StudentLSTM architecture:')
print(model_probe)
print(f'\nTotal parameters     : {total_params:,}')
print(f'Trainable parameters : {trainable_params:,}')

StudentLSTM architecture:
StudentLSTM(
  (lstm): LSTM(7, 64, num_layers=2, batch_first=True, dropout=0.2)
  (classifier): Sequential(
    (0): Linear(in_features=64, out_features=32, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.2, inplace=False)
    (3): Linear(in_features=32, out_features=1, bias=True)
  )
)

Total parameters     : 54,081
Trainable parameters : 54,081


## 4. Training Loop

In [6]:
# ── Hyperparameters and device ────────────────────────────────────────────
EPOCHS    = 30
LR        = 0.001
PATIENCE  = 5
BEST_PATH = RESULTS_DIR / 'lstm_best_model.pth'

# pos_weight: ratio of negative to positive samples (handles 31.2% imbalance)
n_pos = int(y_train.sum())
n_neg = len(y_train) - n_pos
pos_weight = torch.tensor([n_neg / n_pos], dtype=torch.float32)
print(f'pos_weight = {pos_weight.item():.4f}  ({n_neg:,} neg / {n_pos:,} pos)')

# Device selection: prefer MPS (Apple Silicon) > CUDA > CPU
if torch.backends.mps.is_available():
    device = torch.device('mps')
elif torch.cuda.is_available():
    device = torch.device('cuda')
else:
    device = torch.device('cpu')
print(f'Training device: {device}')

pos_weight = 2.2094  (17,940 neg / 8,120 pos)
Training device: mps


In [7]:
# ── Initialise model, optimiser, loss ────────────────────────────────────
model     = StudentLSTM(input_size=N_FEAT).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=LR)
criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight.to(device))

train_losses, val_losses, val_aucs = [], [], []
best_val_auc = 0.0
best_epoch   = 0
no_improve   = 0
best_weights = None

print(f'Starting training for up to {EPOCHS} epochs (early stopping patience={PATIENCE})...')
print(f'{"Epoch":<8} {"TrainLoss":>10} {"ValLoss":>10} {"ValAUC":>9} {"Time":>7}')
print('-' * 50)

t0 = time.time()

for epoch in range(1, EPOCHS + 1):
    # ── Train ────────────────────────────────────────────────────────────
    model.train()
    running_loss = 0.0
    for X_b, y_b in train_loader:
        X_b, y_b = X_b.to(device), y_b.to(device)
        optimizer.zero_grad()
        logits = model(X_b)
        loss   = criterion(logits, y_b)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        running_loss += loss.item() * len(y_b)
    train_loss = running_loss / n_train

    # ── Validate ─────────────────────────────────────────────────────────
    model.eval()
    val_running = 0.0
    all_probs   = []
    with torch.no_grad():
        for X_b, y_b in test_loader:
            X_b, y_b = X_b.to(device), y_b.to(device)
            logits = model(X_b)
            val_running += criterion(logits, y_b).item() * len(y_b)
            all_probs.append(torch.sigmoid(logits).cpu())
    val_loss = val_running / n_test
    val_probs = torch.cat(all_probs).numpy()
    val_auc   = roc_auc_score(y_test, val_probs)

    train_losses.append(train_loss)
    val_losses.append(val_loss)
    val_aucs.append(val_auc)

    # ── Early stopping & checkpoint ────────────────────────────────────
    if val_auc > best_val_auc:
        best_val_auc = val_auc
        best_epoch   = epoch
        no_improve   = 0
        best_weights = copy.deepcopy(model.state_dict())
        torch.save(best_weights, BEST_PATH)
    else:
        no_improve += 1

    if epoch % 5 == 0 or epoch == 1:
        elapsed = time.time() - t0
        marker  = ' *' if epoch == best_epoch else ''
        print(f'Epoch {epoch:3d}/{EPOCHS} | {train_loss:10.4f} | {val_loss:10.4f} | '
              f'{val_auc:9.4f} | {elapsed:6.0f}s{marker}')

    if no_improve >= PATIENCE:
        print(f'\nEarly stopping triggered at epoch {epoch} '
              f'(no improvement for {PATIENCE} epochs).')
        break

total_time = time.time() - t0
print(f'\nTraining complete in {total_time:.1f}s')
print(f'Best epoch: {best_epoch}  |  Best Val AUC: {best_val_auc:.4f}')

Starting training for up to 30 epochs (early stopping patience=5)...
Epoch     TrainLoss    ValLoss    ValAUC    Time
--------------------------------------------------


Epoch   1/30 |     0.4615 |     0.4230 |    0.9315 |      7s *


Epoch   5/30 |     0.4005 |     0.3962 |    0.9409 |     22s *


Epoch  10/30 |     0.3907 |     0.4001 |    0.9405 |     41s


Epoch  15/30 |     0.3853 |     0.3912 |    0.9421 |     59s *


Epoch  20/30 |     0.3776 |     0.3968 |    0.9419 |     78s


Epoch  25/30 |     0.3712 |     0.3946 |    0.9428 |     97s


Epoch  30/30 |     0.3611 |     0.4069 |    0.9422 |    116s

Training complete in 115.8s
Best epoch: 27  |  Best Val AUC: 0.9436


In [8]:
# ── Plot 1: Training history ───────────────────────────────────────────────
epoch_range = range(1, len(train_losses) + 1)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(epoch_range, train_losses, 'b-', linewidth=2, label='Train Loss')
ax1.plot(epoch_range, val_losses,   'r-', linewidth=2, label='Val Loss')
ax1.axvline(best_epoch, color='green', linestyle='--', linewidth=1.5,
            label=f'Best epoch ({best_epoch})')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('BCEWithLogitsLoss')
ax1.set_title('Train vs Validation Loss')
ax1.legend()

ax2.plot(epoch_range, val_aucs, 'g-', linewidth=2, label='Val AUC')
ax2.axvline(best_epoch, color='red', linestyle='--', linewidth=1.5,
            label=f'Best epoch ({best_epoch})')
ax2.axhline(best_val_auc, color='gray', linestyle=':', linewidth=1.2,
            label=f'Best AUC = {best_val_auc:.4f}')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('AUC-ROC')
ax2.set_title('Validation AUC by Epoch')
ax2.legend()

fig.suptitle('LSTM Training History', fontsize=14)
plt.tight_layout()
out1 = FIGURES_DIR / '04_lstm_training.png'
fig.savefig(out1, dpi=DPI, bbox_inches='tight')
plt.close(fig)
print(f'Saved \u2192 {out1}')

Saved → /Users/sergazieff.n/VSCodeProjects/eduforecast/results/figures/04_lstm_training.png


## 5. Evaluation

In [9]:
print('Loading best checkpoint and evaluating on full test set...')

model.load_state_dict(torch.load(BEST_PATH, map_location=device))
model.eval()

all_probs_list = []
with torch.no_grad():
    for X_b, _ in test_loader:
        logits = model(X_b.to(device))
        all_probs_list.append(torch.sigmoid(logits).cpu())

lstm_prob = torch.cat(all_probs_list).numpy()
lstm_pred = (lstm_prob >= 0.5).astype(int)

lstm_acc  = accuracy_score(y_test, lstm_pred)
lstm_prec = precision_score(y_test, lstm_pred, average='weighted', zero_division=0)
lstm_rec  = recall_score(y_test, lstm_pred, average='weighted', zero_division=0)
lstm_f1   = f1_score(y_test, lstm_pred, average='weighted', zero_division=0)
lstm_auc  = roc_auc_score(y_test, lstm_prob)

# Sensitivity: recall for the positive (at-risk) class
lstm_sens = recall_score(y_test, lstm_pred, pos_label=1, average='binary', zero_division=0)

print()
print('LSTM RESULTS')
print(f'  Accuracy    : {lstm_acc:.4f}')
print(f'  Precision   : {lstm_prec:.4f}')
print(f'  Recall      : {lstm_rec:.4f}')
print(f'  F1 Score    : {lstm_f1:.4f}')
print(f'  AUC-ROC     : {lstm_auc:.4f}')
print(f'  Sensitivity : {lstm_sens:.4f}  (recall for At-Risk class)')
print()
print('Classification Report:')
print(classification_report(y_test, lstm_pred, target_names=['Not At-Risk', 'At-Risk']))

Loading best checkpoint and evaluating on full test set...



LSTM RESULTS
  Accuracy    : 0.8564
  Precision   : 0.8894
  Recall      : 0.8564
  F1 Score    : 0.8611
  AUC-ROC     : 0.9436
  Sensitivity : 0.9578  (recall for At-Risk class)

Classification Report:
              precision    recall  f1-score   support

 Not At-Risk       0.98      0.81      0.89      4497
     At-Risk       0.70      0.96      0.81      2036

    accuracy                           0.86      6533
   macro avg       0.84      0.88      0.85      6533
weighted avg       0.89      0.86      0.86      6533



In [10]:
lstm_metrics = {
    'model':     'LSTM',
    'accuracy':  round(lstm_acc,  4),
    'precision': round(lstm_prec, 4),
    'recall':    round(lstm_rec,  4),
    'f1':        round(lstm_f1,   4),
    'auc_roc':   round(lstm_auc,  4),
    'sensitivity': round(lstm_sens, 4),
}
pd.DataFrame([lstm_metrics]).to_csv(METRICS_DIR / 'lstm_metrics.csv', index=False)
print('Saved \u2192 lstm_metrics.csv')

Saved → lstm_metrics.csv


## 6. Temporal AUC Analysis

In [11]:
print('Computing temporal AUC for weeks 1\u201339...')
print('(zero-masking: at week W, future weeks W+1\u201339 are set to zero)')

RF_AUC = 0.9472
LR_AUC = 0.9375

model.eval()
temporal_aucs: list[tuple[int, float]] = []

t_start = time.time()
for w in range(1, 40):
    # Create a masked copy: zero out all weeks > w
    X_masked = X_test_t.clone()       # (n_test, 40, 7)
    X_masked[:, w:, :] = 0.0

    probs_w: list[float] = []
    with torch.no_grad():
        for start in range(0, n_test, 512):
            batch = X_masked[start:start + 512].to(device)
            p = torch.sigmoid(model(batch)).cpu().numpy()
            probs_w.extend(p.tolist())

    auc_w = roc_auc_score(y_test, probs_w)
    temporal_aucs.append((w, auc_w))

print(f'Done in {time.time()-t_start:.1f}s')

# Print key weeks
print(f'\n{"Week":>6}  {"LSTM AUC":>10}  {"vs RF":>9}')
print('-' * 30)
for w, auc_w in temporal_aucs:
    if w in [5, 10, 15, 20, 25, 30, 39]:
        delta = auc_w - RF_AUC
        print(f'Week {w:2d}   {auc_w:.4f}    {delta:+.4f}')

# First week LSTM exceeds RF
exceeds_rf = [(w, a) for w, a in temporal_aucs if a >= RF_AUC]
if exceeds_rf:
    w_cross, a_cross = exceeds_rf[0]
    print(f'\nLSTM first exceeds RF baseline at week {w_cross}  (AUC={a_cross:.4f})')
else:
    w_cross, a_cross = None, None
    print('\nLSTM does not exceed RF AUC at any individual week (uses full sequence)')

Computing temporal AUC for weeks 1–39...
(zero-masking: at week W, future weeks W+1–39 are set to zero)


Done in 4.6s

  Week    LSTM AUC      vs RF
------------------------------
Week  5   0.4294    -0.5178
Week 10   0.2870    -0.6602
Week 15   0.4092    -0.5380
Week 20   0.3557    -0.5915
Week 25   0.3120    -0.6352
Week 30   0.4151    -0.5321
Week 39   0.9021    -0.0451

LSTM does not exceed RF AUC at any individual week (uses full sequence)


In [12]:
# ── Plot 2: Temporal AUC curve ─────────────────────────────────────────────
weeks_list, aucs_list = zip(*temporal_aucs)

fig, ax = plt.subplots(figsize=(12, 6))

ax.plot(weeks_list, aucs_list, 'b-o', markersize=4, linewidth=2.2, label='LSTM')
ax.axhline(RF_AUC, color='red',    linestyle='--', linewidth=1.8,
           label=f'RF Baseline (AUC={RF_AUC})')
ax.axhline(LR_AUC, color='orange', linestyle='--', linewidth=1.8,
           label=f'LR Baseline (AUC={LR_AUC})')

if w_cross is not None:
    ax.axvline(w_cross, color='green', linestyle=':', linewidth=1.5,
               label=f'LSTM \u2265 RF at Week {w_cross}')
    ax.annotate(
        f'Week {w_cross}:\nLSTM exceeds RF',
        xy=(w_cross, a_cross),
        xytext=(w_cross + 1.5, a_cross - 0.025),
        fontsize=9, color='darkgreen',
        arrowprops=dict(arrowstyle='->', color='darkgreen', lw=1.2),
    )

ax.set_xlabel('Week Number', fontsize=12)
ax.set_ylabel('AUC-ROC', fontsize=12)
ax.set_title(
    'LSTM \u2014 Prediction AUC by Week (Early Warning Capability)',
    fontsize=13, pad=12,
)
ax.legend(fontsize=10, loc='lower right')
ax.set_xlim(1, 39)
ax.set_ylim(max(0.45, min(aucs_list) - 0.05), 1.01)
plt.tight_layout()
out2 = FIGURES_DIR / '04_lstm_temporal_auc.png'
fig.savefig(out2, dpi=DPI)
plt.close(fig)
print(f'Saved \u2192 {out2}')

Saved → /Users/sergazieff.n/VSCodeProjects/eduforecast/results/figures/04_lstm_temporal_auc.png


## 7. Updated Model Comparison

In [13]:
# Load baseline results and append LSTM
baseline = pd.read_csv(METRICS_DIR / 'baseline_comparison.csv')

lstm_row = pd.DataFrame([{
    'model':     'LSTM',
    'accuracy':  round(lstm_acc,  4),
    'precision': round(lstm_prec, 4),
    'recall':    round(lstm_rec,  4),
    'f1':        round(lstm_f1,   4),
    'auc_roc':   round(lstm_auc,  4),
}])

comparison_v2 = pd.concat([baseline, lstm_row], ignore_index=True)
comparison_v2.to_csv(METRICS_DIR / 'model_comparison_v2.csv', index=False)
print('Saved \u2192 model_comparison_v2.csv')

rf_auc_base = float(baseline[baseline['model'] == 'Random Forest']['auc_roc'].iloc[0])
auc_delta   = lstm_auc - rf_auc_base

print()
print('=' * 57)
print('MODEL COMPARISON (Updated)')
print('=' * 57)
print(f'{"Model":<25} {"AUC":>7} {"F1":>7} {"Acc":>7}')
print('-' * 57)
for _, row in comparison_v2.iterrows():
    print(f'{row["model"]:<25} {row["auc_roc"]:>7.4f} {row["f1"]:>7.4f} {row["accuracy"]:>7.4f}')
print('=' * 57)
print(f'LSTM vs Random Forest: AUC delta = {auc_delta:+.4f}')
print('=' * 57)

Saved → model_comparison_v2.csv

MODEL COMPARISON (Updated)
Model                         AUC      F1     Acc
---------------------------------------------------------
Logistic Regression        0.9375  0.8596  0.8560
Random Forest              0.9472  0.8629  0.8584
LSTM                       0.9436  0.8611  0.8564
LSTM vs Random Forest: AUC delta = -0.0036


In [14]:
# ── Plot 3: Updated grouped bar chart ─────────────────────────────────────
metric_names   = ['accuracy', 'precision', 'recall', 'f1', 'auc_roc']
display_metrics = ['Accuracy', 'Precision', 'Recall', 'F1', 'AUC-ROC']
model_labels   = list(comparison_v2['model'])
palette        = ['#3498db', '#e74c3c', '#2ecc71']

x     = np.arange(len(metric_names))
n_mdl = len(comparison_v2)
width = 0.22
offsets = np.linspace(-(n_mdl - 1) * width / 2,
                       (n_mdl - 1) * width / 2, n_mdl)

fig, ax = plt.subplots(figsize=(13, 6))
for i, (_, row) in enumerate(comparison_v2.iterrows()):
    vals = [row[m] for m in metric_names]
    bars = ax.bar(x + offsets[i], vals, width,
                  label=model_labels[i], color=palette[i],
                  alpha=0.87, edgecolor='white')
    for bar, val in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width() / 2,
                bar.get_height() + 0.005,
                f'{val:.3f}', ha='center', va='bottom', fontsize=7.5)

ax.set_xticks(x)
ax.set_xticklabels(display_metrics, fontsize=11)
ax.set_ylabel('Score', fontsize=12)
ax.set_ylim(0, 1.12)
ax.set_title('Model Comparison: Baseline vs LSTM', fontsize=14, pad=12)
ax.legend(fontsize=10)
ax.yaxis.set_major_formatter(mticker.FormatStrFormatter('%.2f'))
plt.tight_layout()
out3 = FIGURES_DIR / '04_model_comparison.png'
fig.savefig(out3, dpi=DPI)
plt.close(fig)
print(f'Saved \u2192 {out3}')

print('\nAll figures saved:')
for f in sorted(FIGURES_DIR.glob('04_*.png')):
    print(f'  {f.name}')

Saved → /Users/sergazieff.n/VSCodeProjects/eduforecast/results/figures/04_model_comparison.png

All figures saved:
  04_lstm_temporal_auc.png
  04_lstm_training.png
  04_model_comparison.png
